In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import uuid
import time

In [0]:
metadata_df = spark.table("healthcare.default.metadata_config")

display(metadata_df)

source_id,source_name,file_path,file_format,target_table,primary_key,load_type,active_flag,data_classification,description
SRC001,patients,/Volumes/healthcare/default/healthcare_landing/patients.csv,csv,healthcare.default.bronze_patients,patient_id,FULL,true,PII,Patient master data
SRC002,appointments,/Volumes/healthcare/default/healthcare_landing/appointments.csv,csv,healthcare.default.bronze_appointments,appointment_id,FULL,true,PHI,Patient appointment records
SRC003,billing,/Volumes/healthcare/default/healthcare_landing/billing.csv,csv,healthcare.default.bronze_billing,bill_id,FULL,true,PHI,Hospital billing records
SRC004,doctors,/Volumes/healthcare/default/healthcare_landing/doctors.csv,csv,healthcare.default.bronze_doctors,doctor_id,FULL,true,PII,Doctor master data
SRC005,treatments,/Volumes/healthcare/default/healthcare_landing/treatments.csv,csv,healthcare.default.bronze_treatments,treatment_id,FULL,true,PHI,Patient treatment records


In [0]:
active_sources = (
    metadata_df
    .filter(F.col("active_flag") == True)
)

print("Active sources:", active_sources.count())

display(active_sources)

Active sources: 5


source_id,source_name,file_path,file_format,target_table,primary_key,load_type,active_flag,data_classification,description
SRC001,patients,/Volumes/healthcare/default/healthcare_landing/patients.csv,csv,healthcare.default.bronze_patients,patient_id,FULL,true,PII,Patient master data
SRC002,appointments,/Volumes/healthcare/default/healthcare_landing/appointments.csv,csv,healthcare.default.bronze_appointments,appointment_id,FULL,true,PHI,Patient appointment records
SRC003,billing,/Volumes/healthcare/default/healthcare_landing/billing.csv,csv,healthcare.default.bronze_billing,bill_id,FULL,true,PHI,Hospital billing records
SRC004,doctors,/Volumes/healthcare/default/healthcare_landing/doctors.csv,csv,healthcare.default.bronze_doctors,doctor_id,FULL,true,PII,Doctor master data
SRC005,treatments,/Volumes/healthcare/default/healthcare_landing/treatments.csv,csv,healthcare.default.bronze_treatments,treatment_id,FULL,true,PHI,Patient treatment records


In [0]:
batch_id = f"BATCH_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"

print("Batch ID:", batch_id)

Batch ID: BATCH_20260810_144925_920fea


In [0]:
for row in active_sources.collect():
    print(f"{row.source_name:15} → {row.file_path}")

patients        → /Volumes/healthcare/default/healthcare_landing/patients.csv
appointments    → /Volumes/healthcare/default/healthcare_landing/appointments.csv
billing         → /Volumes/healthcare/default/healthcare_landing/billing.csv
doctors         → /Volumes/healthcare/default/healthcare_landing/doctors.csv
treatments      → /Volumes/healthcare/default/healthcare_landing/treatments.csv


In [0]:
# Bronze ingestion

for row in active_sources.collect():

    source_id = row.source_id
    source_name = row.source_name
    file_path = row.file_path
    target_table = row.target_table

    print(f"\nProcessing: {source_name}")
    print(f"Source: {file_path}")
    print(f"Target: {target_table}")

    start_time = time.time()

    try:
        # Read CSV
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(file_path)
        )

        rows_read = df.count()

        # Add Bronze metadata
        bronze_df = (
            df
            .withColumn("_batch_id", F.lit(batch_id))
            .withColumn("_source_id", F.lit(source_id))
            .withColumn("_source_name", F.lit(source_name))
            .withColumn("_source_file_name", F.lit(file_path.split("/")[-1]))
            .withColumn("_ingestion_timestamp", F.current_timestamp())
            .withColumn("_ingestion_date", F.current_date())
            .withColumn(
                "_record_hash",
                F.sha2(
                    F.concat_ws(
                        "||",
                        *[
                            F.coalesce(F.col(c).cast("string"), F.lit(""))
                            for c in df.columns
                        ]
                    ),
                    256
                )
            )
            .withColumn("_layer", F.lit("BRONZE"))
        )

        # Write Bronze Delta table
        (
            bronze_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )

        rows_written = bronze_df.count()

        duration = time.time() - start_time

        print(f"Rows read: {rows_read}")
        print(f"Rows written: {rows_written}")
        print(f"Status: SUCCESS")
        print(f"Duration: {duration:.2f} seconds")

    except Exception as e:

        duration = time.time() - start_time

        print(f"Status: FAILED")
        print(f"Error: {str(e)}")


Processing: patients
Source: /Volumes/healthcare/default/healthcare_landing/patients.csv
Target: healthcare.default.bronze_patients
Rows read: 50
Rows written: 50
Status: SUCCESS
Duration: 4.17 seconds

Processing: appointments
Source: /Volumes/healthcare/default/healthcare_landing/appointments.csv
Target: healthcare.default.bronze_appointments
Rows read: 200
Rows written: 200
Status: SUCCESS
Duration: 3.12 seconds

Processing: billing
Source: /Volumes/healthcare/default/healthcare_landing/billing.csv
Target: healthcare.default.bronze_billing
Rows read: 200
Rows written: 200
Status: SUCCESS
Duration: 3.32 seconds

Processing: doctors
Source: /Volumes/healthcare/default/healthcare_landing/doctors.csv
Target: healthcare.default.bronze_doctors
Rows read: 10
Rows written: 10
Status: SUCCESS
Duration: 3.22 seconds

Processing: treatments
Source: /Volumes/healthcare/default/healthcare_landing/treatments.csv
Target: healthcare.default.bronze_treatments
Rows read: 200
Rows written: 200
Status

In [0]:
bronze_tables = [
    "healthcare.default.bronze_patients",
    "healthcare.default.bronze_appointments",
    "healthcare.default.bronze_billing",
    "healthcare.default.bronze_doctors",
    "healthcare.default.bronze_treatments"
]

for table_name in bronze_tables:
    row_count = spark.table(table_name).count()
    print(f"{table_name}: {row_count} rows")

healthcare.default.bronze_patients: 50 rows
healthcare.default.bronze_appointments: 200 rows
healthcare.default.bronze_billing: 200 rows
healthcare.default.bronze_doctors: 10 rows
healthcare.default.bronze_treatments: 200 rows


In [0]:
display(
    spark.table("healthcare.default.bronze_patients")
)

patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email,_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_record_hash,_layer
P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com,BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,ba212534f7335cf7916442c841c05750cf45d3949e54a83671efc04a9fec017b,BRONZE
P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com,BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,b0d0ac80f2431a01436014cedf9e734686c274b3a5f2504d91fe246aba3ecc48,BRONZE
P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com,BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,8ef41b5779c7f1823805464b7466cf586728b24925577e11b16cd6af39974cc7,BRONZE
P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com,BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,632ec24cae21e04c7d9a1057e9c975b3bc0d2ffe362c1713979229b46824602d,BRONZE
P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com,BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,621118d37d68abe8e6785477555c55e06253be7c353855f790d1cd0b23aea3ec,BRONZE
P006,Linda,Jones,M,1963-06-16,7561777264,321 Maple Dr,2022-10-02,HealthIndia,INS613758,linda.jones@mail.com,BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,98a54635c5e5b77e690311de03f3dd6cbd2a0af8c20d3381513311f480710afa,BRONZE
P007,Alex,Johnson,F,1989-06-08,6278710077,789 Pine Rd,2021-12-25,MedCare Plus,INS465890,alex.johnson@mail.com,BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,08d33f0571ac6307f0531f077f04e3d9f02ecf29ac066d29b293ba6b900f5ec4,BRONZE
P008,David,Davis,F,1976-07-05,7090558393,456 Oak Ave,2021-05-25,WellnessCorp,INS545101,david.davis@mail.com,BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,dbc9d33e90d090ff0ab1d4dd39722bb67368f4f0450209c67ecd7f94809a99d5,BRONZE
P009,Laura,Davis,M,1971-12-11,7060324619,321 Maple Dr,2022-09-18,PulseSecure,INS136631,laura.davis@mail.com,BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,1e47aecc86b23f2385d88498f101c0b8493d8d9c53d20721a0d7bd97b155b585,BRONZE
P010,Michael,Taylor,M,2001-10-13,7081396733,123 Elm St,2022-08-24,WellnessCorp,INS866577,michael.taylor@mail.com,BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,79f33e68feadb9a5f626fb4ef2e969bdbb9c63914c221394d309d242c4142833,BRONZE


In [0]:
display(
    spark.table("healthcare.default.bronze_patients")
    .select(
        "_batch_id",
        "_source_id",
        "_source_name",
        "_source_file_name",
        "_ingestion_timestamp",
        "_ingestion_date",
        "_layer"
    )
    .limit(10)
)

_batch_id,_source_id,_source_name,_source_file_name,_ingestion_timestamp,_ingestion_date,_layer
BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,BRONZE
BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,BRONZE
BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,BRONZE
BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,BRONZE
BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,BRONZE
BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,BRONZE
BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,BRONZE
BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,BRONZE
BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,BRONZE
BATCH_20260810_144925_920fea,SRC001,patients,patients.csv,2026-08-10T14:49:29.152Z,2026-08-10,BRONZE


In [0]:
display(
    spark.table("healthcare.default.bronze_patients")
    .select(
        "patient_id",
        "_record_hash"
    )
    .limit(10)
)

patient_id,_record_hash
P001,ba212534f7335cf7916442c841c05750cf45d3949e54a83671efc04a9fec017b
P002,b0d0ac80f2431a01436014cedf9e734686c274b3a5f2504d91fe246aba3ecc48
P003,8ef41b5779c7f1823805464b7466cf586728b24925577e11b16cd6af39974cc7
P004,632ec24cae21e04c7d9a1057e9c975b3bc0d2ffe362c1713979229b46824602d
P005,621118d37d68abe8e6785477555c55e06253be7c353855f790d1cd0b23aea3ec
P006,98a54635c5e5b77e690311de03f3dd6cbd2a0af8c20d3381513311f480710afa
P007,08d33f0571ac6307f0531f077f04e3d9f02ecf29ac066d29b293ba6b900f5ec4
P008,dbc9d33e90d090ff0ab1d4dd39722bb67368f4f0450209c67ecd7f94809a99d5
P009,1e47aecc86b23f2385d88498f101c0b8493d8d9c53d20721a0d7bd97b155b585
P010,79f33e68feadb9a5f626fb4ef2e969bdbb9c63914c221394d309d242c4142833


In [0]:
# Record the successful Bronze ingestion in the audit log

audit_records = []

for row in active_sources.collect():

    source_name = row.source_name
    source_id = row.source_id
    classification = row.data_classification
    target_table = row.target_table

    # Get actual row count from Bronze table
    rows_written = spark.table(target_table).count()

    audit_records.append({
        "audit_id": str(uuid.uuid4()),
        "batch_id": batch_id,
        "source_id": source_id,
        "source_name": source_name,
        "layer": "BRONZE",
        "pipeline_start_time": None,
        "pipeline_end_time": None,
        "rows_read": rows_written,
        "rows_written": rows_written,
        "rows_rejected": 0,
        "rows_quarantined": 0,
        "status": "SUCCESS",
        "error_message": None,
        "pipeline_duration_secs": None,
        "notebook_name": "01_ingest_bronze",
        "environment": "Databricks Free Edition",
        "dq_score_avg": None,
        "sla_met": None,
        "retry_attempt": 0,
        "data_classification": classification,
        "downstream_notified": False
    })


# Get the exact schema from our existing audit table
audit_schema = spark.table(
    "healthcare.default.audit_log"
).schema


# Create DataFrame using the explicit schema
audit_df = spark.createDataFrame(
    audit_records,
    schema=audit_schema
)


# Add pipeline timestamps
audit_df = (
    audit_df
    .withColumn(
        "pipeline_start_time",
        F.current_timestamp()
    )
    .withColumn(
        "pipeline_end_time",
        F.current_timestamp()
    )
)


# Write audit records
audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("healthcare.default.audit_log")


print("Audit records written successfully.")
print("Records added:", audit_df.count())

Audit records written successfully.
Records added: 5


In [0]:
display(
    spark.table("healthcare.default.audit_log")
    .select(
        "source_name",
        "layer",
        "rows_read",
        "rows_written",
        "rows_rejected",
        "rows_quarantined",
        "status",
        "batch_id"
    )
    .orderBy("source_name")
)

source_name,layer,rows_read,rows_written,rows_rejected,rows_quarantined,status,batch_id
appointments,BRONZE,200,200,0,0,SUCCESS,BATCH_20260810_144925_920fea
billing,BRONZE,200,200,0,0,SUCCESS,BATCH_20260810_144925_920fea
doctors,BRONZE,10,10,0,0,SUCCESS,BATCH_20260810_144925_920fea
patients,BRONZE,50,50,0,0,SUCCESS,BATCH_20260810_144925_920fea
treatments,BRONZE,200,200,0,0,SUCCESS,BATCH_20260810_144925_920fea
